## PDF extraction

In [ ]:
import contextlib
from collections.abc import Iterable
from pathlib import Path

from docling_core.types.doc import ImageRefMode
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.pipeline_options import (
    EasyOcrOptions,
    HeadingHierarchyOptions,
    PdfPipelineOptions,
    PictureDescriptionVlmOptions,
    TableFormerMode,
    TableStructureOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption


class Pdf2DoclingDocument:
    """Convert native PDFs to DoclingDocuments (headings, tables, captioned pictures, OCR).

    Unlike jpg files, where we build the Docling
    Document ourselves, PDFs go through Docling's
    automatic pipeline. A PDF page mixes text, headings, tables and images, and rebuilding the
    layout analysis, reading order, table structure and picture extraction by hand would be
    far more complex than configuring the existing pipeline.

    Known limitations observed on our corpus:
      - Picture classification and description are not reliable: some images are not
        classified or described, and descriptions can be inaccurate.
      - Table extraction has issues: some tables come out incomplete or badly structured,
        so important tables should be checked against the original PDF.
      - OCR does not seem to extract the text inside images. In the default OCR mode, OCR
        runs on regions found by the layout model that lack native PDF text, so text in an
        image that is not detected as its own region, or that overlaps native text, may be
        missed.
    """

    DEFAULT_IMAGE_PROMPT = (
        "Décris cette image extraite d'un document français, pour qu'une personne qui ne la voit pas "
        "comprenne son contenu. Réponds uniquement en français, sans introduction, sans rien inventer.\n\n"
        "Réponds exactement dans ce format Markdown, sans titre (#) :\n\n"
        "**Type :** schéma, organigramme, graphique, tableau, logo, icône, capture d'écran, "
        "palette de couleurs, photographie ou autre\n\n"
        "**Description :** selon le type :\n"
        "- schéma ou organigramme : une liste à puces, un élément ou un lien par ligne (« A → B »), "
        "dans l'ordre des étapes ;\n"
        "- graphique : le type, les axes, les séries, les valeurs lisibles et la tendance ;\n"
        "- tableau : un tableau Markdown reprenant les colonnes et les lignes ;\n"
        "- palette de couleurs : un tableau Markdown | Couleur | Code |, une ligne par couleur ;\n"
        "- logo ou icône : sa forme et ses couleurs, en une ou deux phrases ;\n"
        "- capture d'écran ou maquette : une liste à puces des zones visibles et de leur contenu ;\n"
        "- photographie : les personnes, objets, lieu, action et ambiance, en quelques phrases.\n\n"
    )

    def __init__(
        self,
        artifacts_path: str | Path,
        vlm_model: str,
        image_prompt: str | None = None,
        ocr_langs: list[str] | None = None,
        num_threads: int = 4,
    ) -> None:
        # Docling loads the description model from "artifacts_path / repo_id":
        opts = PdfPipelineOptions(artifacts_path=artifacts_path)

        # recover PDF heading levels
        # source: https://docling-project.github.io/docling/usage/advanced_options/#recover-pdf-heading-levels
        opts.heading_hierarchy_options = HeadingHierarchyOptions(enabled=True)
        opts.generate_parsed_pages = True  # required by the font-style signal

        # rebuild tables as real rows and columns instead of loose text
        # source: https://docling-project.github.io/docling/reference/pipeline_options/#docling.datamodel.pipeline_options.TableStructureOptions.do_cell_matching
        opts.do_table_structure = True
        opts.table_structure_options = TableStructureOptions(do_cell_matching=True)
        opts.table_structure_options.mode = TableFormerMode.ACCURATE  # more accurate TableFormer model

        # extract images from the pdf, describe them for the retrieval system and the LLM
        # source: https://docling-project.github.io/docling/_generated/examples/export_figures/
        # source: https://docling-project.github.io/docling/usage/enrichments/#picture-classification
        opts.generate_picture_images = True
        opts.images_scale = 2
        opts.do_picture_classification = False  # classification with the picture description model
        opts.do_picture_description = True
        opts.picture_description_options = PictureDescriptionVlmOptions(
            repo_id=vlm_model,
            prompt=image_prompt or self.DEFAULT_IMAGE_PROMPT,
            generation_config={"max_new_tokens": 512, "do_sample": False},  # default truncates long descriptions
        )

        # TODO: check if ocr is working
        # read text that exists only as pixels (labels inside diagrams, scanned inserts)
        # source: https://docling-project.github.io/docling/_generated/examples/custom_convert/
        opts.do_ocr = True
        opts.ocr_options = EasyOcrOptions(lang=ocr_langs or ["fr"])

        # use the GPU if available, otherwise CPU threads
        # source: https://docling-project.github.io/docling/_generated/examples/custom_convert/
        opts.accelerator_options = AcceleratorOptions(num_threads=num_threads, device=AcceleratorDevice.AUTO)

        self.converter = DocumentConverter(
            format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
        )

    def convert_all(self, paths: Iterable[str | Path], out_dir: str | Path) -> None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        for result in self.converter.convert_all(paths, raises_on_error=False):
            name = Path(result.input.file).stem.strip()

            if result.status == ConversionStatus.FAILURE:
                print("FAILED:", name, result.errors)
                continue

            doc = result.document
            with contextlib.chdir(out_dir):  # keeps the image folder next to the files, not nested
                doc.save_as_json(Path(f"{name}.json"), image_mode=ImageRefMode.REFERENCED)
                doc.save_as_markdown(Path(f"{name}.md"), image_mode=ImageRefMode.REFERENCED)
            print("SAVED:", name, "|", len(doc.tables), "tables |", len(doc.pictures), "pictures")

In [ ]:
from pathlib import Path

from docling_core.types.doc import DoclingDocument

from extract_pdf import Pdf2DoclingDocument


# PDF convertion
pdf_paths = sorted(Path("data/INJA/Livrable 1/Donnée entrée").rglob("*.pdf", case_sensitive=False))
pdf_converter = Pdf2DoclingDocument(artifacts_path="/linkhome/idris/genini01/ssos027/.cache/docling/models",
                                    vlm_model="Qwen3-VL-32B-Instruct")
pdf_converter.convert_all(pdf_paths, "data/INJA/Livrable 1/DoclingConvertion/pdf")

# Import converted pdf
imported_doc = DoclingDocument.load_from_json(Path(f"data/INJA/Livrable 1/DoclingConvertion/pdf/{pdf_paths[0].stem.strip()}.json"))
print(imported_doc.export_to_markdown())

## JPG extraction

In [ ]:
import contextlib
from collections.abc import Iterable
from pathlib import Path

from docling_core.types.doc import (
    DoclingDocument, DocItemLabel, ImageRef, ImageRefMode,
    ProvenanceItem, BoundingBox, Size, CoordOrigin,
)
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText


class Jpg2DoclingDocument:
    """Turn a jpg photo into a one-page DoclingDocument whose picture is captioned by a VLM.

    Why we build the DoclingDocument ourselves instead of using Docling's DocumentConverter:
    when we fed jpg files to Docling's image pipeline, it did not reliably detect the photo
    as a picture item, so there was no picture to caption or describe. Our working hypothesis
    is that its layout model is designed for document pages (text, tables, figures) and does
    not treat a whole photograph as a figure; we did not confirm this.

    Since the output must still be in the same DoclingDocument format as the other file types,
    we build it directly:
      - one page, sized like the image;
      - one picture covering that whole page (provenance = full-page bounding box);
      - one caption, generated by a VLM (transformers), linked to the picture.
    """

    DEFAULT_PROMPT = (
        "Décris cette image extraite d'un document français, pour qu'une personne qui ne la voit pas "
        "comprenne son contenu. Réponds uniquement en français, sans introduction, sans rien inventer.\n\n"
        "Réponds exactement dans ce format Markdown, sans titre (#) :\n\n"
        "**Type :** schéma, organigramme, graphique, tableau, logo, icône, capture d'écran, "
        "palette de couleurs, photographie ou autre\n\n"
        "**Description :** selon le type :\n"
        "- schéma ou organigramme : une liste à puces, un élément ou un lien par ligne (« A → B »), "
        "dans l'ordre des étapes ;\n"
        "- graphique : le type, les axes, les séries, les valeurs lisibles et la tendance ;\n"
        "- tableau : un tableau Markdown reprenant les colonnes et les lignes ;\n"
        "- palette de couleurs : un tableau Markdown | Couleur | Code |, une ligne par couleur ;\n"
        "- logo ou icône : sa forme et ses couleurs, en une ou deux phrases ;\n"
        "- capture d'écran ou maquette : une liste à puces des zones visibles et de leur contenu ;\n"
        "- photographie : les personnes, objets, lieu, action et ambiance, en quelques phrases.\n\n"
    )

    def __init__(
        self,
        vlm_model: str | Path,
        prompt: str | None = None,
        max_side: int = 1280,
        max_new_tokens: int = 512,
    ) -> None:
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.processor = AutoProcessor.from_pretrained(vlm_model)
        self.model = AutoModelForImageTextToText.from_pretrained(vlm_model).to(self.device)
        self.prompt = prompt or self.DEFAULT_PROMPT
        self.max_side = max_side
        self.max_new_tokens = max_new_tokens

    def describe(self, img: Image.Image) -> str:
        input_img = img.copy()
        input_img.thumbnail((self.max_side, self.max_side))  # limit img size
        chat = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": self.prompt}]}]
        chat_prompt = self.processor.apply_chat_template(chat, add_generation_prompt=True)
        inputs = self.processor(images=input_img, text=chat_prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            # for descriptions we want faithful, reproducible text, not variety
            output_ids = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens, do_sample=False)
        new_ids = output_ids[0][inputs["input_ids"].shape[1]:]  # drop the prompt tokens
        return self.processor.decode(new_ids, skip_special_tokens=True).strip()

    def convert(self, path: str | Path) -> DoclingDocument:
        path = Path(path)
        img = Image.open(path).convert("RGB")
        w, h = img.size

        doc = DoclingDocument(name=path.stem)
        doc.add_page(page_no=1, size=Size(width=w, height=h))  # empty page with img size
        caption = doc.add_text(label=DocItemLabel.CAPTION, text=self.describe(img))  # add caption
        doc.add_picture(
            image=ImageRef.from_pil(img, dpi=72),
            caption=caption,
            prov=ProvenanceItem(
                page_no=1, charspan=(0, 0),
                bbox=BoundingBox(l=0, t=0, r=w, b=h, coord_origin=CoordOrigin.TOPLEFT),
            ),
        )  # add img with reference to caption
        return doc

    def convert_all(self, paths: Iterable[str | Path], out_dir: str | Path) -> None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        for path in map(Path, paths):
            doc = self.convert(path)
            with contextlib.chdir(out_dir):  # keeps the image folder next to the files, not nested
                doc.save_as_json(Path(f"{path.stem.strip()}.json"), image_mode=ImageRefMode.REFERENCED)
                doc.save_as_markdown(Path(f"{path.stem.strip()}.md"), image_mode=ImageRefMode.REFERENCED)
            print("SAVED:", path.name)

In [ ]:
from pathlib import Path

from docling_core.types.doc import DoclingDocument

from extract_jpg import Jpg2DoclingDocument


# JPG convertion
jpg_paths = sorted(Path("data/INJA/Livrable 1/Donnée entrée").rglob("*.jpg", case_sensitive=False))
img_converter = Jpg2DoclingDocument(vlm_model="/lustre/fsmisc/dataset/HuggingFace_Models/Qwen/Qwen3-VL-32B-Instruct")
img_converter.convert_all(jpg_paths, "data/INJA/Livrable 1/DoclingConvertion/jpg")

# Import converted jpg
imported_doc = DoclingDocument.load_from_json(Path(f"data/INJA/Livrable 1/DoclingConvertion/jpg/{jpg_paths[0].stem.strip()}.json"))
print(imported_doc.export_to_markdown())

## PPTX extraction

In [ ]:
import contextlib
from collections.abc import Iterable
from pathlib import Path

from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.document import ConversionResult
from docling.document_converter import DocumentConverter


class SimplePptx2DoclingDocument:
    """Convert text-only pptx files to DoclingDocuments with Docling's default PPTX converter."""

    def __init__(self) -> None:
        self.converter = DocumentConverter(allowed_formats=[InputFormat.PPTX])

    def convert_all(self, paths: Iterable[str | Path], out_dir: str | Path) -> None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        for result in self.converter.convert_all(paths, raises_on_error=False):
            name = Path(result.input.file).stem.strip()

            if result.status == ConversionStatus.FAILURE:
                print("FAILED:", name, result.errors)
                continue

            doc = result.document
            with contextlib.chdir(out_dir):  # keeps the image folder next to the files, not nested
                doc.save_as_json(f"{name}.json")
                doc.save_as_markdown(f"{name}.md")
            print("SAVED:", name, "|", len(doc.pages), "slides")

In [ ]:
from pathlib import Path

from docling_core.types.doc import DoclingDocument

from extract_pptx import SimplePptx2DoclingDocument


# PPTX convertion
pptx_paths = sorted(Path("data/INJA/Livrable 1/Donnée entrée").rglob("*.pptx", case_sensitive=False))
pptx_converter = SimplePptx2DoclingDocument()
pptx_converter.convert_all(pptx_paths, "data/INJA/Livrable 1/DoclingConvertion/pptx")

# Import converted pptx
imported_doc = DoclingDocument.load_from_json(Path(f"data/INJA/Livrable 1/DoclingConvertion/pptx/{pptx_paths[0].stem.strip()}.json"))
print(imported_doc.export_to_markdown())

## XLSX extraction

In [ ]:
from collections.abc import Iterable
from pathlib import Path

import pandas as pd
import torch
from docling_core.types.doc import DocItemLabel
from docling.datamodel.base_models import InputFormat, ConversionStatus
from docling.document_converter import DocumentConverter
from transformers import AutoProcessor, AutoModelForImageTextToText


class Xlsx2DoclingDocument:
    """Convertit des fichiers Excel en DoclingDocument, avec une description de chaque tableau.

    Le tableau lui-même est extrait par le DocumentConverter de Docling (backend XLSX), et
    non reconstruit à la main. Ce choix est volontaire : Docling lit directement la structure
    du classeur (lignes, colonnes, cellules fusionnées, donc catégories de colonnes sur
    plusieurs niveaux) et produit les cellules du tableau. Le reconstruire nous-mêmes depuis
    pandas obligerait à gérer à la main les en-têtes multi-niveaux, les spans de colonnes et
    les types de valeurs, avec un risque d'erreur plus élevé.

    pandas n'est utilisé que pour lire les premières lignes brutes de chaque feuille
    (header=None, pas d'interprétation d'en-tête). Ces lignes sont données à un modèle de
    vision-langage (texte seul) qui produit une description en français, ajoutée au tableau
    comme légende (caption). Le tableau complet reste celui de Docling.
    """
    DEFAULT_PROMPT = (
        "Voici les premières lignes brutes d'une feuille Excel (« {sheet} »). "
        "Les premières lignes sont des en-têtes, parfois sur "
        "plusieurs niveaux : une catégorie de colonnes peut couvrir plusieurs colonnes, "
        "et n'apparaît alors qu'au-dessus de la première.\n\n"
        "{table}\n\n"
        "Décris en français, en 3 à 5 phrases : le sujet du tableau, les catégories de colonnes, "
        "le rôle de chaque colonne et ce que représente une ligne. "
        "N'invente rien, ne donne aucun total ni statistique."
    )

    def __init__(self, vlm_model: str | Path, n_head: int = 10, max_new_tokens: int = 512) -> None:
        self.converter = DocumentConverter(allowed_formats=[InputFormat.XLSX])
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.processor = AutoProcessor.from_pretrained(vlm_model)
        self.model = AutoModelForImageTextToText.from_pretrained(vlm_model).to(self.device)
        self.n_head = n_head
        self.max_new_tokens = max_new_tokens

    def describe(self, head: pd.DataFrame, sheet: str) -> str:
        prompt = self.DEFAULT_PROMPT.format(
            sheet=sheet, table=head.fillna("").to_string(header=False, index=False))
        chat = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        chat_prompt = self.processor.apply_chat_template(chat, add_generation_prompt=True)
        inputs = self.processor(text=chat_prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            # for descriptions we want faithful, reproducible text, not variety
            output_ids = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens, do_sample=False)
        new_ids = output_ids[0][inputs["input_ids"].shape[1]:]  # drop the prompt tokens
        return self.processor.decode(new_ids, skip_special_tokens=True).strip()

    def convert_all(self, paths: Iterable[str | Path], out_dir: str | Path) -> None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        for result in self.converter.convert_all(paths, raises_on_error=False):
            path = Path(result.input.file)
            name = path.stem.strip()

            if result.status == ConversionStatus.FAILURE:
                print("FAILED:", name, result.errors)
                continue

            doc = result.document
            heads = pd.read_excel(path, sheet_name=None, header=None, nrows=self.n_head, dtype=object)
            for sheet_no, (sheet, head) in enumerate(heads.items(), start=1):
                table = next((t for t in doc.tables if t.prov and t.prov[0].page_no == sheet_no), None)
                if table is None:
                    continue
                caption = doc.add_text(label=DocItemLabel.CAPTION, text=self.describe(head, sheet))
                table.captions.append(caption.get_ref())

            doc.save_as_json(out_dir / f"{name}.json")
            doc.save_as_markdown(out_dir / f"{name}.md")
            print("SAVED:", name, "|", len(doc.tables), "tables")

In [ ]:
from pathlib import Path

from docling_core.types.doc import DoclingDocument

from extract_xlsx import Xlsx2DoclingDocument


# XLSX convertion
xlsx_paths = sorted(Path("data/INJA/Livrable 1/Donnée entrée").rglob("*.xlsx", case_sensitive=False))
xlsx_converter = Xlsx2DoclingDocument(vlm_model="/lustre/fsmisc/dataset/HuggingFace_Models/Qwen/Qwen3-VL-32B-Instruct")
xlsx_converter.convert_all(xlsx_paths, "data/INJA/Livrable 1/DoclingConvertion/xlsx")

# Import converted xlsx
imported_doc = DoclingDocument.load_from_json(Path(f"data/INJA/Livrable 1/DoclingConvertion/xlsx/{xlsx_paths[0].stem.strip()}.json"))
print(imported_doc.export_to_markdown())

## DOCX extraction

In [ ]:
import contextlib
from collections.abc import Iterable
from pathlib import Path

from docling_core.types.doc import ImageRefMode
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat, ConversionStatus
from docling.datamodel.pipeline_options import ConvertPipelineOptions, PictureDescriptionVlmOptions
from docling.document_converter import DocumentConverter, WordFormatOption


class Docx2DoclingDocument:
    """Convert Word (.docx) files to DoclingDocuments, with classified and described pictures.

    DOCX files go through Docling's automatic pipeline. Unlike a PDF, a DOCX already stores
    its structure (headings, paragraphs, lists, tables), so Docling's Word backend reads it
    directly through the simple pipeline: no layout model, no table structure model, no OCR.
    Rebuilding this ourselves would add complexity without improving the result.

    Pictures: the simple pipeline inherits the enrichment step of Docling's ConvertPipeline,
    so picture classification and picture description (VLM, same prompt style as for PDFs)
    run on the images embedded in the document.
    """

    DEFAULT_IMAGE_PROMPT = (
        "Décris cette image extraite d'un document français, pour qu'une personne qui ne la voit pas "
        "comprenne son contenu. Réponds uniquement en français, sans introduction, sans rien inventer.\n\n"
        "Réponds exactement dans ce format Markdown, sans titre (#) :\n\n"
        "**Type :** schéma, organigramme, graphique, tableau, logo, icône, capture d'écran, "
        "palette de couleurs, photographie ou autre\n\n"
        "**Description :** selon le type :\n"
        "- schéma ou organigramme : une liste à puces, un élément ou un lien par ligne (« A → B »), "
        "dans l'ordre des étapes ;\n"
        "- graphique : le type, les axes, les séries, les valeurs lisibles et la tendance ;\n"
        "- tableau : un tableau Markdown reprenant les colonnes et les lignes ;\n"
        "- palette de couleurs : un tableau Markdown | Couleur | Code |, une ligne par couleur ;\n"
        "- logo ou icône : sa forme et ses couleurs, en une ou deux phrases ;\n"
        "- capture d'écran ou maquette : une liste à puces des zones visibles et de leur contenu ;\n"
        "- photographie : les personnes, objets, lieu, action et ambiance, en quelques phrases.\n\n"
    )

    def __init__(self,
                 artifacts_path: str | Path,
                 vlm_model: str,
                 image_prompt: str | None = None,
                 num_threads: int = 4) -> None:
        # Docling loads the description model from "artifacts_path / repo_id":
        opts = ConvertPipelineOptions(
            artifacts_path=artifacts_path,
            accelerator_options=AcceleratorOptions(num_threads=num_threads, device=AcceleratorDevice.AUTO),
            do_picture_classification=True,
            do_picture_description=True,
            picture_description_options=PictureDescriptionVlmOptions(
                repo_id=vlm_model,
                prompt=image_prompt or self.DEFAULT_IMAGE_PROMPT,
                generation_config={"max_new_tokens": 512, "do_sample": False},  # default truncates long descriptions
            ),
        )
        self.converter = DocumentConverter(
            allowed_formats=[InputFormat.DOCX],
            format_options={InputFormat.DOCX: WordFormatOption(pipeline_options=opts)},
        )

    def convert_all(self, paths: Iterable[str | Path], out_dir: str | Path) -> None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        for result in self.converter.convert_all(paths, raises_on_error=False):
            name = Path(result.input.file).stem.strip()

            if result.status == ConversionStatus.FAILURE:
                print("FAILED:", name, result.errors)
                continue

            doc = result.document
            with contextlib.chdir(out_dir):  # keeps the image folder next to the files, not nested
                doc.save_as_json(Path(f"{name}.json"), image_mode=ImageRefMode.REFERENCED)
                doc.save_as_markdown(Path(f"{name}.md"), image_mode=ImageRefMode.REFERENCED)
            print("SAVED:", name, "|", len(doc.tables), "tables |", len(doc.pictures), "pictures")

In [ ]:
from pathlib import Path

from docling_core.types.doc import DoclingDocument

from extract_docx import Docx2DoclingDocument


# DOCX convertion
docx_paths = sorted(Path("data/INJA/Livrable 1/Donnée entrée").rglob("*.docx", case_sensitive=False))
docx_converter = Docx2DoclingDocument(artifacts_path="/linkhome/idris/genini01/ssos027/.cache/docling/models",
                                      vlm_model="Qwen3-VL-32B-Instruct")
docx_converter.convert_all(docx_paths, "data/INJA/Livrable 1/DoclingConvertion/docx")

# Import converted docx
imported_doc = DoclingDocument.load_from_json(Path(f"data/INJA/Livrable 1/DoclingConvertion/docx/{docx_paths[0].stem.strip()}.json"))
print(imported_doc.export_to_markdown())

## VLM extraction

We use a VLM to extract the content of the pdf.

In [ ]:
import contextlib
from collections.abc import Iterable
from pathlib import Path

from docling_core.types.doc import ImageRefMode
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat, ConversionStatus
from docling.datamodel.pipeline_options import VlmPipelineOptions
from docling.datamodel.pipeline_options_vlm_model import (
    InferenceFramework, InlineVlmOptions, ResponseFormat, TransformersModelType,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.pipeline.vlm_pipeline import VlmPipeline


class VlmPdf2DoclingDocument:
    """Convert PDFs to DoclingDocuments with Docling's VLM pipeline (one VLM reads each page image).

    A single vision-language model transcribes each rendered page (text, tables, images),
    and Docling turns its output (DocTags or Markdown) into a DoclingDocument.
    """

    # source: https://github.com/docling-project/docling/blob/main/docling/datamodel/vlm_model_specs.py#L274
    DEFAULT_PAGE_PROMPT = "Convert this page to markdown. Do not miss any text and only output the bare markdown!"

    def __init__(self,
                 artifacts_path: str | Path,
                 vlm_model: str,
                 vlm_options: InlineVlmOptions | None = None,
                 num_threads: int = 4) -> None:
        # Docling loads the VLM from "artifacts_path / repo_id":
        vlm_options = (vlm_options or InlineVlmOptions(
            repo_id="",  # set below
            prompt=self.DEFAULT_PAGE_PROMPT,
            response_format=ResponseFormat.MARKDOWN,
            inference_framework=InferenceFramework.TRANSFORMERS,
            transformers_model_type=TransformersModelType.AUTOMODEL_IMAGETEXTTOTEXT,
            max_new_tokens=4096,
        )).model_copy(update={"repo_id": vlm_model})

        opts = VlmPipelineOptions(
            artifacts_path=artifacts_path,
            accelerator_options=AcceleratorOptions(num_threads=num_threads, device=AcceleratorDevice.AUTO),
            vlm_options=vlm_options,
            # the VLM always sees the rendered pages; these flags only decide what is kept in the output
            generate_page_images=False,
            generate_picture_images=False,
        )
        self.converter = DocumentConverter(
            allowed_formats=[InputFormat.PDF],
            format_options={InputFormat.PDF: PdfFormatOption(pipeline_cls=VlmPipeline, pipeline_options=opts)},
        )

    def convert_all(self, paths: Iterable[str | Path], out_dir: str | Path) -> None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        for result in self.converter.convert_all(paths, raises_on_error=False):
            name = Path(result.input.file).stem.strip()

            if result.status == ConversionStatus.FAILURE:
                print("FAILED:", name, result.errors)
                continue

            doc = result.document
            with contextlib.chdir(out_dir):  # keeps the image folder next to the files, not nested
                doc.save_as_json(Path(f"{name}.json"), image_mode=ImageRefMode.REFERENCED)
                doc.save_as_markdown(Path(f"{name}.md"), image_mode=ImageRefMode.REFERENCED)
            print("SAVED:", name, "|", doc.num_pages(), "pages |", len(doc.tables), "tables |", len(doc.pictures), "pictures")

In [ ]:
from pathlib import Path

from docling_core.types.doc import DoclingDocument

from extract_pdf_vlm import VlmPdf2DoclingDocument


# PDF convertion
pdf_paths = sorted(Path("data/INJA/Livrable 1/Donnée entrée").rglob("*.pdf", case_sensitive=False))
pdf_converter = VlmPdf2DoclingDocument(artifacts_path="/linkhome/idris/genini01/ssos027/.cache/docling/models",
                                       vlm_model="Qwen3-VL-32B-Instruct")
pdf_converter.convert_all(pdf_paths, "data/INJA/Livrable 1/DoclingConvertion/pdf_vision")

# Import converted pdf
imported_doc = DoclingDocument.load_from_json(Path(f"data/INJA/Livrable 1/DoclingConvertion/pdf_vision/{pdf_paths[0].stem.strip()}.json"))
print(imported_doc.export_to_markdown())

## **RAPTOR**

In [ ]:
import json
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import torch
import umap
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams


@dataclass
class Node:
    id: int  # position in self.nodes; chunk ids cited in the syntheses are level-0 node ids
    text: str
    level: int  # 0 = original chunk, 1+ = summary
    theme: str = ""  # theme given by the LLM to a summary; empty for original chunks
    children: list[int] = field(default_factory=list)

class RaptorThemeSynthesizer:
    """Theme-level synthesis of a corpus: group the chunks into themes with a RAPTOR tree
    (Sarthi et al., 2024), then write one synthesis per theme by map-reduce over the original chunks.

    Tree: embed texts, reduce them with UMAP, cluster them with a Gaussian mixture (global then local,
    soft membership), summarize each cluster with an LLM, and repeat on the summaries.
    Source: https://github.com/langchain-ai/langchain/blob/v0.1.17rc1/cookbook/RAPTOR.ipynb
    
    Themes: the top-level nodes are merged when near-duplicate; each theme is synthesized from the
    original chunks under it.

    LLM calls go through vLLM: all the clusters of a tree level are sent at once, and each theme
    is synthesized in one call (plus one merge call if needed). The output is constrained by a
    JSON schema (structured outputs), so the format is always respected.
    """

    SUMMARY_PROMPT = (
        "Voici des extraits de documents.\n\n"
        "<extraits>\n{texts}\n</extraits>\n\n"
        "Résume ces extraits en français, uniquement à partir de leur contenu. "
        "Le résumé doit faire environ un quart de la longueur des extraits, entrer "
        "directement dans le sujet sans introduction et conserver tous les faits "
        "utiles : chiffres, dates, noms, titres de documents. N'invente rien.\n\n"
        "Réponds avec deux champs :\n"
        "- `theme` : le thème principal des extraits ;\n"
        "- `summary` : le résumé."
    )
    
    THEME_SYNTHESIS_PROMPT = (
        "Voici des extraits de documents portant sur un même thème.\n"
        "Chaque extrait est précédé de son identifiant entre crochets.\n\n"
        "<extraits>\n{chunks}\n</extraits>\n\n"
        "Rédige en français une synthèse de ce que ces extraits disent sur ce thème, "
        "uniquement à partir de leur contenu. Regroupe ce qui se répète, relie ce qui "
        "se complète et signale ce qui varie ou se contredit d'un document à l'autre. "
        "Conserve tous les faits distincts : chiffres, dates, noms, titres de documents. "
        "N'invente rien.\n\n"
        "Réponds avec quatre champs :\n"
        "- `title` : un titre court pour le thème ;\n"
        "- `opening` : un paragraphe d'ouverture de 2 à 3 phrases donnant le message d'ensemble ;\n"
        "- `points` : la liste des faits distincts, chacun avec `fact` (le fait) et "
        "`sources` (les identifiants des extraits qui l'appuient) ;\n"
        "- `divergences` : les contradictions entre extraits. Une divergence existe seulement si deux "
        "affirmations ne peuvent pas être vraies en même temps : des valeurs, dates ou noms différents pour "
        "la même chose, ou des positions opposées sur une même question. Des valeurs identiques ne sont pas "
        "une divergence. Regroupe en une seule divergence les écarts qui portent sur un même ensemble (par "
        "exemple toutes les différences d'une même grille de critères), et ne répète jamais une divergence. "
        "Ne sont PAS des divergences : qu'un extrait ne mentionne pas un fait ; qu'un document approuve, "
        "reprend, confirme ou complète ce que dit un autre ; une simple différence de formulation. "
        "Liste vide s'il n'y en a pas."
    )

    MERGE_PROMPT = (
        "Voici plusieurs synthèses partielles sur un même thème.\n"
        "Elles portent sur des extraits différents d'un même ensemble de documents.\n\n"
        "<syntheses>\n{parts}\n</syntheses>\n\n"
        "Fusionne-les en français en une seule synthèse, uniquement à partir de leur contenu. "
        "Rédige un nouveau paragraphe d'ouverture, sans recopier ceux des synthèses partielles. "
        "Regroupe les faits identiques en un seul, en gardant tous leurs identifiants, et conserve "
        "tous les faits distincts : chiffres, dates, noms, titres de documents. Reprends les "
        "divergences des synthèses partielles en regroupant les identiques, et ajoute les "
        "contradictions entre faits de parties différentes. N'invente rien.\n\n"
        "Réponds avec quatre champs :\n"
        "- `title` : un titre court pour le thème ;\n"
        "- `opening` : un paragraphe d'ouverture de 2 à 3 phrases donnant le message d'ensemble ;\n"
        "- `points` : la liste des faits distincts, chacun avec `fact` (le fait) et "
        "`sources` (les identifiants des extraits qui l'appuient) ;\n"
        "- `divergences` : les contradictions entre extraits. Une divergence existe seulement si deux "
        "affirmations ne peuvent pas être vraies en même temps : des valeurs, dates ou noms différents pour "
        "la même chose, ou des positions opposées sur une même question. Des valeurs identiques ne sont pas "
        "une divergence. Regroupe en une seule divergence les écarts qui portent sur un même ensemble (par "
        "exemple toutes les différences d'une même grille de critères), et ne répète jamais une divergence. "
        "Ne sont PAS des divergences : qu'un extrait ne mentionne pas un fait ; qu'un document approuve, "
        "reprend, confirme ou complète ce que dit un autre ; une simple différence de formulation. "
        "Liste vide s'il n'y en a pas."
    )

    MAX_POINTS = 20  # above this, split_themes divides a theme into sub-themes

    SPLIT_PROMPT = (
        "Voici les points d'une synthèse sur le thème : {title}\n"
        "Un point par ligne, au format JSON : `fact` (le fait) et `sources` (les identifiants des extraits "
        "qui l'appuient).\n\n"
        "<points>\n{points}\n</points>\n\n"
        "Voici ses divergences, au même format :\n\n"
        "<divergences>\n{divergences}\n</divergences>\n\n"
        "Cette synthèse a trop de points pour être lue d'un coup. Répartis ces points en sous-thèmes selon "
        "le sens, comme les sections d'une synthèse destinée à un lecteur : chaque sous-thème porte sur un "
        "sujet distinct, et chaque point va dans un seul sous-thème. Choisis le nombre de sous-thèmes que "
        "demande le contenu. Puis rédige pour chaque sous-thème une synthèse succincte de ses points : "
        "regroupe les points proches en un seul, {max_points} points au plus par sous-thème. N'invente rien.\n\n"
        "Réponds avec un champ `sub_themes` : les sous-thèmes, chacun avec :\n"
        "- `title` : un titre court ;\n"
        "- `opening` : une ou deux phrases donnant l'essentiel du sous-thème ;\n"
        "- `points` : les idées essentielles, chacune avec `fact` (l'idée, en une phrase) et `sources` "
        "(les identifiants des extraits des points regroupés) ;\n"
        "- `divergences` : les divergences ci-dessus qui concernent ce sous-thème, chacune avec `fact` et "
        "`sources`. Une divergence est un cas où au moins deux extraits donnent des informations "
        "incompatibles sur un même point (valeurs, dates, noms ou affirmations différentes) ; qu'un extrait "
        "ne mentionne pas un fait n'est PAS une divergence. Liste vide s'il n'y en a pas."
    )

    SUMMARY_SCHEMA = {
        "type": "object",
        "properties": {"theme": {"type": "string"}, "summary": {"type": "string"}},
        "required": ["theme", "summary"],
        "additionalProperties": False,
    }

    SOURCED_FACT_SCHEMA = {
        "type": "object",
        "properties": {
            "fact": {"type": "string"},
            "sources": {"type": "array", "minItems": 1, "items": {"type": "integer"}},
        },
        "required": ["fact", "sources"],
        "additionalProperties": False,
    }

    SYNTHESIS_SCHEMA = {
        "type": "object",
        "properties": {
            "title": {"type": "string"},
            "opening": {"type": "string"},
            "points": {"type": "array", "minItems": 1, "items": SOURCED_FACT_SCHEMA},
            "divergences": {"type": "array", "maxItems": MAX_POINTS, "items": SOURCED_FACT_SCHEMA},
        },
        "required": ["title", "opening", "points", "divergences"],
        "additionalProperties": False,
    }

    SPLIT_SCHEMA = {
        "type": "object",
        "properties": {
            "sub_themes": {"type": "array", "minItems": 2, "items": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "opening": {"type": "string"},
                    "points": {"type": "array", "minItems": 1, "maxItems": MAX_POINTS, "items": SOURCED_FACT_SCHEMA},
                    "divergences": {"type": "array", "maxItems": MAX_POINTS, "items": SOURCED_FACT_SCHEMA},
                },
                "required": ["title", "opening", "points", "divergences"],
                "additionalProperties": False,
            }},
        },
        "required": ["sub_themes"],
        "additionalProperties": False,
    }

    def __init__(self, embed_model: str | Path, llm_model: str | Path,
                 reduced_dim: int = 10, threshold: float = 0.1, max_clusters: int = 50, local_neighbors: int = 10,
                 n_levels: int = 2, max_input_tokens: int = 16384, max_new_tokens: int = 32768,
                 tensor_parallel_size: int = 1, gpu_memory_utilization: float = 0.95,
                 random_state: int = 224) -> None:
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.llm = LLM(
            model=str(llm_model),
            dtype="bfloat16",
            tensor_parallel_size=tensor_parallel_size,
            gpu_memory_utilization=gpu_memory_utilization,
            max_model_len=max_input_tokens + max_new_tokens,
            max_num_batched_tokens=max_input_tokens + max_new_tokens,
            max_num_seqs=256,
            trust_remote_code=False,
        )  # vLLM reserves gpu_memory_utilization of the GPU, the embedder uses the rest
        self.tokenizer = self.llm.get_tokenizer()
        self.embedder = SentenceTransformer(str(embed_model), device=self.device)
        self.reduced_dim = reduced_dim
        self.threshold = threshold
        self.max_clusters = max_clusters
        self.local_neighbors = local_neighbors
        self.n_levels = n_levels
        self.max_input_tokens = max_input_tokens
        self.max_new_tokens = max_new_tokens
        self.random_state = random_state

        # output storage
        self.nodes: list[Node] = []
        self.embeddings = np.empty((0, 0))

    ## LLM (vLLM, guided decoding) -----------------------------------------------------------------------------------

    def _generate_json(self, prompts: list[str], schemas: list[dict]) -> list[dict]:
        """Run all prompts in one vLLM batch; each output is constrained by its own JSON schema."""
        messages = [[{"role": "user", "content": prompt}] for prompt in prompts]
        params = [SamplingParams(temperature=0.3, top_p=0.9,
                                 max_tokens=self.max_new_tokens,
                                 structured_outputs=StructuredOutputsParams(json=schema))
                  for schema in schemas]
        outputs = self.llm.chat(messages, params)

        input_lengths = [len(o.prompt_token_ids) for o in outputs]
        print(f"{len(outputs)} requests, input tokens: max {max(input_lengths)}, total {sum(input_lengths)}")

        results = []
        for output in outputs:
            generation = output.outputs[0]
            if generation.finish_reason != "stop":  # truncated JSON cannot be parsed
                tail = generation.text[-3000:]
                raise RuntimeError(f"Generation ended with '{generation.finish_reason}' after "
                                   f"{len(generation.token_ids)} tokens "
                                   f"(input {len(output.prompt_token_ids)} tokens). Tail:\n{tail!r}")
            results.append(json.loads(generation.text))
        return results

    def _count_tokens(self, text: str) -> int:
        return len(self.tokenizer.encode(text, add_special_tokens=False))

    ## Clustering -----------------------------------------------------------------------------------------------------

    def _gmm_bic(self, reduced: np.ndarray, k: int) -> float:
        """BIC of a k-component Gaussian mixture fitted on the reduced embeddings (lower is better)."""
        return GaussianMixture(k, random_state=self.random_state).fit(reduced).bic(reduced)

    def _gmm_soft_clusters(self, reduced: np.ndarray) -> list[list[int]]:
        """Pick the cluster count by BIC, then keep every text whose probability exceeds the threshold.

        Example with 4 texts, best_k = 3 and threshold = 0.1:
            probs (texts × clusters)        output
            text 0: [0.95, 0.04, 0.01]      [[0, 2],   # cluster 0: texts 0 and 2
            text 1: [0.02, 0.97, 0.01]       [1, 2],   # cluster 1: texts 1 and 2
            text 2: [0.60, 0.35, 0.05]       [3]]      # cluster 2: text 3
            text 3: [0.01, 0.02, 0.97]
        """
        bics = [self._gmm_bic(reduced, k) for k in range(1, min(self.max_clusters, len(reduced)))]
        best_k = int(np.argmin(bics)) + 1  # bics[0] is k=1
        probs = GaussianMixture(best_k, random_state=self.random_state).fit(reduced).predict_proba(reduced)
        return [np.flatnonzero(probs[:, c] > self.threshold).tolist() for c in range(best_k)]

    def _reduce_dimensions(self, embeddings: np.ndarray, n_neighbors: int) -> np.ndarray:
        n_neighbors = max(2, min(n_neighbors, len(embeddings) - 1))  # UMAP needs 2 <= n_neighbors < n
        return umap.UMAP(n_neighbors=n_neighbors, n_components=self.reduced_dim, metric="cosine",
                         random_state=self.random_state).fit_transform(embeddings)

    def _cluster_global_local(self, embeddings: np.ndarray) -> list[list[int]]:
        """Global clustering, then local clustering inside each global cluster. Returns lists of indices."""
        if len(embeddings) <= self.reduced_dim + 1:
            return [list(range(len(embeddings)))]  # too few texts for UMAP dim reduction

        clusters = []
        global_reduced = self._reduce_dimensions(embeddings, int((len(embeddings) - 1) ** 0.5))
        for members in self._gmm_soft_clusters(global_reduced):
            if len(members) == 0:
                continue
            if len(members) <= self.reduced_dim + 1:
                clusters.append(members)  # keep the small global cluster as it is
                continue
            local_reduced = self._reduce_dimensions(embeddings[members], self.local_neighbors)
            for local_positions in self._gmm_soft_clusters(local_reduced):
                if len(local_positions) == 0:
                    continue  # empty local cluster
                # map local positions (within this global cluster) to global positions in `embeddings`
                clusters.append([members[p] for p in local_positions])
        return clusters

    ## Raptor tree summary --------------------------------------------------------------------------------------------

    def embed_texts(self, texts: list[str]) -> np.ndarray:
        return self.embedder.encode(texts, normalize_embeddings=True, batch_size=8)
    
    def build_tree(self, texts: list[str], cluster_texts: list[str] | None = None) -> None:
        """texts: read by the LLM and indexed; cluster_texts: used to cluster the leaves (default: texts)."""
        self.nodes = [Node(id=i, text=text, level=0) for i, text in enumerate(texts)]
    
        current_nodes = list(self.nodes)  # nodes of the level being clustered
        for level in range(1, self.n_levels + 1):
            # Cluster the current level
            texts_to_embed = (cluster_texts if (cluster_texts and level == 1)
                              else [node.text for node in current_nodes])
            clusters_indices = self._cluster_global_local(self.embed_texts(texts_to_embed))
            print(f"level {level}: {len(current_nodes)} nodes -> {len(clusters_indices)} clusters "
                  f"(sizes: {sorted(len(indices) for indices in clusters_indices)})")
    
            # Summarize all the clusters of the level in one vLLM batch
            clusters_children = [[current_nodes[i] for i in indices] for indices in clusters_indices]
            prompts = []
            for children in clusters_children:
                prompts.append(
                    self.SUMMARY_PROMPT.format(
                        texts="\n\n---\n\n".join([child.text for child in children])
                    )
                )
            summaries = self._generate_json(prompts, [self.SUMMARY_SCHEMA] * len(prompts))

            parent_nodes = []
            for children, summary in zip(clusters_children, summaries):
                parent = Node(id=len(self.nodes), text=summary["summary"].strip(), level=level,
                              theme=summary["theme"].strip(), children=[child.id for child in children])
                self.nodes.append(parent)
                parent_nodes.append(parent)
    
            current_nodes = parent_nodes
            if len(clusters_indices) == 1:
                break  # a single cluster: its summary is the root
    
        self.embeddings = self.embed_texts([node.text for node in self.nodes])

    def search_tree(self, query: str, k: int = 5) -> list[tuple[int, float]]:
        """Collapsed-tree retrieval: returns (node id, cosine score) over chunks and summaries of every level."""
        scores = self.embeddings @ self.embed_texts([query])[0]
        node_ids_by_score = np.argsort(scores)[::-1]          # node ids sorted by score, best first
        top_ids = node_ids_by_score[:k]                       # keep the k best nodes
        return [(int(i), float(scores[i])) for i in top_ids]  # (node id, score) pairs

    def leaf_chunk_ids(self, node_id: int) -> list[int]:
        """All original chunks under a node (a leaf returns itself)."""
        node = self.nodes[node_id]  # a node's id is its position in self.nodes
        if node.level == 0:
            return [node.id]

        leaves = set()
        for child in node.children:                  # for each child of this summary
            for leaf in self.leaf_chunk_ids(child):  # take all the leaves under that child
                leaves.add(leaf)                     # add them; a set ignores duplicates
        return sorted(leaves)                        # back to a list, in corpus order

    def save_tree(self, out_dir: str | Path) -> None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        nodes_as_dicts = [asdict(n) for n in self.nodes]
        json_text = json.dumps(nodes_as_dicts, ensure_ascii=False, indent=2)
        (out_dir / "nodes.json").write_text(json_text, encoding="utf-8")
        np.save(out_dir / "embeddings.npy", self.embeddings)

    def load_tree(self, in_dir: str | Path) -> None:
        in_dir = Path(in_dir)

        json_text = (in_dir / "nodes.json").read_text(encoding="utf-8")
        self.nodes = [Node(**d) for d in json.loads(json_text)]
        self.embeddings = np.load(in_dir / "embeddings.npy")

        if any(n.id != i for i, n in enumerate(self.nodes)):
            raise ValueError("node ids do not match their position in nodes.json")

        if len(self.embeddings) != len(self.nodes):
            raise ValueError(f"{len(self.nodes)} nodes but {len(self.embeddings)} embeddings: "
                             "the files do not come from the same build")

    ## Themes and synthesis -------------------------------------------------------------------------------------------

    def _group_into_themes(self, level: int, distance_threshold: float) -> list[dict]:
        """Merge near-duplicate nodes of `level` into themes."""
        layer = [n for n in self.nodes if n.level == level]
        if len(layer) < 2:  # a single top-level node is a single theme (AgglomerativeClustering needs 2 samples)
            labels = np.zeros(len(layer), dtype=int)
        else:
            E = self.embeddings[[n.id for n in layer]]
            labels = AgglomerativeClustering(n_clusters=None, metric="cosine", linkage="average",
                                             distance_threshold=distance_threshold).fit_predict(E)
        out = []
        for lab in sorted(set(labels)):
            members = [layer[j].id for j in np.flatnonzero(labels == lab)]
            leaves = set().union(*(self.leaf_chunk_ids(m) for m in members))
            out.append({"members": members, "leaves": leaves})
        return out

    def _pack(self, texts: list[str]) -> list[list[str]]:
        """Split texts into batches of at most max_input_tokens (an oversized text gets its own batch)."""
        batches, current, size = [], [], 0
        for text in texts:
            n = self._count_tokens(text)
            if current and size + n > self.max_input_tokens:
                batches.append(current)
                current, size = [], 0
            current.append(text)
            size += n
        batches.append(current)
        return batches

    def synthesize_themes(self, out_dir: str | Path | None = None, level: int = 2,
                          distance_threshold: float = 0.20) -> list[dict]:
        """Merge the top-level nodes into themes, then write one synthesis per theme from its original chunks.

        For each theme: its chunks are split into batches synthesized in one vLLM call (map),
        then the partial syntheses are merged if there are several (reduce).
        If out_dir is given, each theme is saved there as theme_<n>.json."""
        themes = self._group_into_themes(level, distance_threshold)
        if out_dir:
            out_dir = Path(out_dir)
            out_dir.mkdir(parents=True, exist_ok=True)

        for n, th in enumerate(themes, 1):
            leaves = set(th["leaves"])

            # Map: one partial synthesis per batch of chunks
            chunks = [f"[{i}] {self.nodes[i].text}" for i in sorted(leaves)]
            prompts = [self.THEME_SYNTHESIS_PROMPT.format(chunks="\n\n".join(batch))
                       for batch in self._pack(chunks)]
            partials = self._generate_json(prompts, [self.SYNTHESIS_SCHEMA] * len(prompts))

            # Reduce: merge the partial syntheses if there are several
            if len(partials) > 1:
                parts = "\n\n".join(f"[Partie {k + 1}]\n{json.dumps(p, ensure_ascii=False)}"
                                    for k, p in enumerate(partials))
                merge_prompt = self.MERGE_PROMPT.format(parts=parts)
                synthesis = self._generate_json([merge_prompt], [self.SYNTHESIS_SCHEMA])[0]
            else:
                synthesis = partials[0]

            # Result
            for item in synthesis["points"] + synthesis["divergences"]:  # drop ids outside the theme
                item["sources"] = [s for s in item["sources"] if s in leaves]
            cited = {s for item in synthesis["points"] + synthesis["divergences"] for s in item["sources"]}
            th["title"] = synthesis["title"]
            th["opening"] = synthesis["opening"]
            th["points"] = synthesis["points"]
            th["divergences"] = synthesis["divergences"]
            th["leaves"] = cited
            th["unused"] = sorted(leaves - cited)

            if out_dir:
                theme_json = {"title": th["title"], "members": th["members"], "leaves": sorted(th["leaves"]),
                              "unused": th["unused"], "opening": th["opening"],
                              "points": th["points"], "divergences": th["divergences"]}
                (out_dir / f"theme_{n}.json").write_text(
                    json.dumps(theme_json, ensure_ascii=False, indent=2), encoding="utf-8")
        return themes

    @staticmethod
    def load_themes(in_dir: str | Path) -> list[dict]:
        """Load the themes saved by synthesize_themes, in theme order."""
        files = sorted(Path(in_dir).glob("theme_*.json"), key=lambda p: int(p.stem.split("_")[1]))
        themes = []
        for f in files:
            th = json.loads(f.read_text(encoding="utf-8"))
            th["leaves"] = set(th["leaves"])  # same type as in memory (JSON stores a list)
            themes.append(th)
        return themes

    def split_themes(self, theme_dir: str | Path) -> list[dict]:
        """Re-read the themes saved by synthesize_themes and divide those with more than MAX_POINTS points.

        In one LLM call per theme (all themes in one vLLM batch), the points of the theme synthesis are split
        into sub-themes by meaning, and each sub-theme gets a succinct synthesis of at most MAX_POINTS points
        citing chunks, with the divergences that concern it. Every theme file gets a "sub_themes" field
        (empty if the theme was small enough) and "sub_unused": the chunks cited by the theme but by none
        of its sub-themes."""
        themes = self.load_themes(theme_dir)
        big = [th for th in themes if len(th["points"]) > self.MAX_POINTS]
        prompts = [self.SPLIT_PROMPT.format(
                       title=th["title"], max_points=self.MAX_POINTS,
                       points="\n".join(json.dumps({"fact": p["fact"], "sources": p["sources"]}, ensure_ascii=False)
                                        for p in th["points"]),
                       divergences="\n".join(json.dumps({"fact": d["fact"], "sources": d["sources"]}, ensure_ascii=False)
                                             for d in th["divergences"]))
                   for th in big]
        answers = self._generate_json(prompts, [self.SPLIT_SCHEMA] * len(prompts)) if big else []

        for th in themes:
            th["sub_themes"], th["sub_unused"] = [], []
        for th, answer in zip(big, answers):
            for sub in answer["sub_themes"]:
                for key in ("points", "divergences"):  # drop duplicate ids and ids outside the theme
                    for item in sub[key]:
                        item["sources"] = [s for s in dict.fromkeys(item["sources"]) if s in th["leaves"]]
                    sub[key] = [item for item in sub[key] if item["sources"]]
                if sub["points"]:
                    th["sub_themes"].append(sub)
            cited = {s for sub in th["sub_themes"] for key in ("points", "divergences")
                     for item in sub[key] for s in item["sources"]}
            th["sub_unused"] = sorted(th["leaves"] - cited)  # chunks the sub-themes dropped

        files = sorted(Path(theme_dir).glob("theme_*.json"), key=lambda p: int(p.stem.split("_")[1]))
        for f, th in zip(files, themes):
            f.write_text(json.dumps({**th, "leaves": sorted(th["leaves"])}, ensure_ascii=False, indent=2),
                         encoding="utf-8")
        return themes

In [ ]:
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer

from chunking import chunk_documents

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained("/linkhome/idris/genini01/ssos027/.cache/docling/models/harrier-oss-v1-0.6b"),
    max_tokens=512,
)
chunker = HybridChunker(tokenizer=tokenizer)
texts, texts_without_title, sources = chunk_documents("data/INJA/Livrable 1/DoclingConvertion", chunker)

In [ ]:
from raptor import RaptorThemeSynthesizer

raptor_synthesizer = RaptorThemeSynthesizer(
    embed_model="/linkhome/idris/genini01/ssos027/.cache/docling/models/harrier-oss-v1-0.6b",
    llm_model="/lustre/fsmisc/dataset/HuggingFace_Models/Qwen/Qwen3.8-27B",
    tensor_parallel_size=2
)
raptor_synthesizer.build_tree(texts, texts_without_title)
raptor_synthesizer.save_tree("data/INJA/Livrable 1/raptor")

In [ ]:
raptor_synthesizer.load_tree("data/INJA/Livrable 1/raptor")
raptor_synthesizer.synthesize_themes(out_dir="data/INJA/Livrable 1/raptor")
raptor_synthesizer.split_themes(theme_dir="data/INJA/Livrable 1/raptor")

## **GraphRAG**

In [ ]:
import json
from dataclasses import asdict, dataclass
from pathlib import Path

from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams


@dataclass
class EntityNode:
    name: str
    type: str
    description: str
    chunk_id: int  # chunk the entity was extracted from (index in the chunk list)

@dataclass
class Relation:
    source: str  # source entity name
    target: str  # target entity name
    label: str   # short relation name, e.g. "critique"
    description: str
    chunk_id: int


class GraphRAGExtractor:
    """Extract entities and relations from text chunks with a local LLM (first step of GraphRAG).

    Equivalent of the LlamaIndex GraphRAG v1 cookbook's GraphRAGExtractor, without LlamaIndex:
    Source: https://developers.llamaindex.ai/python/examples/cookbooks/graphrag_v1/
    For each chunk, the LLM returns entities (name, type, description) and relations (source, target,
    relation, description) as JSON. Each extracted item keeps the id of its chunk, to cite sources later.
    LLM calls go through vLLM: chunks are sent in batches, and the output is constrained by a JSON schema
    (structured outputs), so the format is always respected.
    """

    EXTRACT_PROMPT = (
        "À partir du texte ci-dessous, extrais au plus {max_knowledge_triplets} relations entre entités.\n"
        "1. Identifie les entités : organismes, personnes, lieux, documents, projets ou dispositifs, notions clés. "
        "Pour chacune : son nom, son type et une description en une phrase.\n"
        "2. Identifie les relations entre ces entités. Pour chacune : l'entité source, l'entité cible, "
        "un nom de relation court (par exemple « finance », « critique », « propose ») et une description "
        "en une phrase.\n"
        "Reprends exactement les noms des entités dans les relations. N'invente rien.\n"
        "Réponds uniquement avec un JSON valide, sans texte autour, au format :\n"
        '{{"entities": [{{"entity_name": "...", "entity_type": "...", "entity_description": "..."}}], '
        '"relationships": [{{"source_entity": "...", "target_entity": "...", "relation": "...", '
        '"relationship_description": "..."}}]}}\n\n'
        "Texte :\n{text}"
    )

    ENTITY_SCHEMA = {
        "type": "object",
        "properties": {
            "entity_name": {"type": "string"},
            "entity_type": {"type": "string"},
            "entity_description": {"type": "string"},
        },
        "required": ["entity_name", "entity_type", "entity_description"],
        "additionalProperties": False,
    }

    RELATION_SCHEMA = {
        "type": "object",
        "properties": {
            "source_entity": {"type": "string"},
            "target_entity": {"type": "string"},
            "relation": {"type": "string"},
            "relationship_description": {"type": "string"},
        },
        "required": ["source_entity", "target_entity", "relation", "relationship_description"],
        "additionalProperties": False,
    }

    def __init__(self, llm_model: str | Path, extract_prompt: str | None = None,
                 max_paths_per_chunk: int = 20, max_input_tokens: int = 8192, max_new_tokens: int = 8192,
                 batch_size: int = 64, tensor_parallel_size: int = 1, gpu_memory_utilization: float = 0.90) -> None:
        self.llm = LLM(
            model=str(llm_model),
            dtype="bfloat16",
            tensor_parallel_size=tensor_parallel_size,
            gpu_memory_utilization=gpu_memory_utilization,
            max_model_len=max_input_tokens + max_new_tokens,
            max_num_seqs=256,
            trust_remote_code=False,
        )
        self.extract_prompt = extract_prompt or self.EXTRACT_PROMPT
        self.max_paths_per_chunk = max_paths_per_chunk
        self.max_new_tokens = max_new_tokens
        self.batch_size = batch_size  # chunks per vLLM call; results are saved after each batch

        # the relation limit of the prompt is also enforced by the schema
        self.extract_schema = {
            "type": "object",
            "properties": {
                "entities": {"type": "array", "maxItems": 2 * max_paths_per_chunk,  # anti-loop guard
                             "items": self.ENTITY_SCHEMA},
                "relationships": {"type": "array", "maxItems": max_paths_per_chunk,
                                  "items": self.RELATION_SCHEMA},
            },
            "required": ["entities", "relationships"],
            "additionalProperties": False,
        }

    def extract_from_llm(self, texts: list[str]) -> list[dict]:
        """Run the extraction prompt on all texts in one vLLM batch; returns the parsed JSON answers."""
        messages = [[{"role": "user",
                      "content": self.extract_prompt.format(text=text, max_knowledge_triplets=self.max_paths_per_chunk)}]
                    for text in texts]
        # for extraction we want faithful, reproducible output, not variety
        params = SamplingParams(temperature=0.0, max_tokens=self.max_new_tokens,
                                structured_outputs=StructuredOutputsParams(json=self.extract_schema))
        outputs = self.llm.chat(messages, params)

        answers = []
        for output in outputs:
            generation = output.outputs[0]
            if generation.finish_reason != "stop":  # truncated JSON cannot be parsed
                tail = generation.text[-3000:]
                raise RuntimeError(f"Generation ended with '{generation.finish_reason}' after "
                                   f"{len(generation.token_ids)} tokens "
                                   f"(input {len(output.prompt_token_ids)} tokens). Tail:\n{tail!r}")
            answers.append(json.loads(generation.text))
        return answers

    def extract_batch(self, chunk_ids: list[int], texts: list[str]) -> list[tuple[list[EntityNode], list[Relation]]]:
        """Extract the entities and relations of several chunks in one vLLM call."""
        results = []
        for chunk_id, answer in zip(chunk_ids, self.extract_from_llm(texts)):
            # the schema guarantees the keys; drop items whose names are empty strings
            entities = [EntityNode(e["entity_name"], e["entity_type"], e["entity_description"], chunk_id)
                        for e in answer["entities"] if e["entity_name"]]
            relations = [Relation(r["source_entity"], r["target_entity"], r["relation"],
                                  r["relationship_description"], chunk_id)
                         for r in answer["relationships"] if r["source_entity"] and r["target_entity"]]
            results.append((entities, relations))
        return results

    def extract(self, chunk_id: int, text: str) -> tuple[list[EntityNode], list[Relation]]:
        """Extract the entities and relations of one chunk."""
        return self.extract_batch([chunk_id], [text])[0]

    def extract_all(self, texts: list[str], out_file: str | Path) -> tuple[list[EntityNode], list[Relation]]:
        """Extract from every chunk, batch_size chunks per vLLM call, saving each result as one JSON line;
        already extracted chunks are skipped, so an interrupted run resumes where it stopped (at worst one batch
        is redone). Chunk ids are positions in `texts` (same order as `infos`)."""
        out_file = Path(out_file)
        done = {json.loads(line)["chunk_id"] for line in out_file.open(encoding="utf-8")} if out_file.exists() else set()
        todo = [chunk_id for chunk_id in range(len(texts)) if chunk_id not in done]

        with out_file.open("a", encoding="utf-8") as f:
            for start in range(0, len(todo), self.batch_size):
                chunk_ids = todo[start:start + self.batch_size]
                results = self.extract_batch(chunk_ids, [texts[i] for i in chunk_ids])
                for chunk_id, (entities, relations) in zip(chunk_ids, results):
                    f.write(json.dumps({"chunk_id": chunk_id,
                                        "entities": [asdict(e) for e in entities],
                                        "relations": [asdict(r) for r in relations]}, ensure_ascii=False) + "\n")
                f.flush()  # written to disk after each batch, not at the end
                n_entities = sum(len(e) for e, _ in results)
                n_relations = sum(len(r) for _, r in results)
                print(f"chunks {len(done) + start + len(chunk_ids)}/{len(texts)}: "
                      f"{n_entities} entities, {n_relations} relations in this batch")
        return self.load(out_file)

    @staticmethod
    def load(out_file: str | Path) -> tuple[list[EntityNode], list[Relation]]:
        """Read back every extraction saved by extract_all."""
        entities, relations = [], []
        for line in Path(out_file).open(encoding="utf-8"):
            row = json.loads(line)
            entities += [EntityNode(**e) for e in row["entities"]]
            relations += [Relation(**r) for r in row["relations"]]
        return entities, relations

In [ ]:
from collections import Counter
from pathlib import Path

from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer

from chunking import chunk_documents
from graphrag_extrator import GraphRAGExtractor


# 1. Chunking
tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained("/linkhome/idris/genini01/ssos027/.cache/docling/models/harrier-oss-v1-0.6b"),
    max_tokens=512,
)
chunker = HybridChunker(tokenizer=tokenizer)
texts, _, sources = chunk_documents("data/INJA/Livrable 1/DoclingConvertion", chunker)
print(len(texts), "chunks")

# 2. Extraction
OUT_DIR = Path("data/INJA/Livrable 1/graphrag")
OUT_DIR.mkdir(parents=True, exist_ok=True)

extractor = GraphRAGExtractor(
    llm_model="/lustre/fsmisc/dataset/HuggingFace_Models/Qwen/Qwen3.8-27B",
    max_paths_per_chunk=40,
    batch_size=64,           # chunks per vLLM call; results are saved after each batch
)
entities, relations = extractor.extract_all(texts, OUT_DIR / "extractions.jsonl")

# 3. Quick checks
print(len(entities), "entity mentions |", len(relations), "relation mentions")

empty = len(texts) - len({e.chunk_id for e in entities})
print(empty, "chunks without any entity")  # the JSON is guaranteed by the schema: these chunks really have none

print("most frequent entities:", Counter(e.name for e in entities).most_common(10))

if relations:
    r = relations[0]
    print(f"{r.source} --{r.label}--> {r.target} | {sources[r.chunk_id].document}, p. {sources[r.chunk_id].pages}")

In [ ]:
from __future__ import annotations  # EntityNode and Relation type hints are not evaluated at import

import json
from collections import Counter, defaultdict
from pathlib import Path

import networkx as nx
from graspologic.partition import hierarchical_leiden
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams


class GraphRAGStore:
    """Build the knowledge graph from extracted mentions, detect communities and synthesize them.

    Equivalent of the LlamaIndex GraphRAG v1 cookbook's GraphRAGStore, without LlamaIndex:
    Source: https://developers.llamaindex.ai/python/examples/cookbooks/graphrag_v1/
    Mentions of the same entity (same name, case-insensitive) become one node; repeated relations between
    the same two entities become one weighted edge. Communities: hierarchical Leiden (graspologic), as in the
    cookbook and the GraphRAG paper; the finest level is kept, then communities that are too small (below
    min_community_size entities, or below min_relative_size of the biggest) are dropped. One succinct LLM
    synthesis per community (title, opening, at most MAX_POINTS essential points and divergences citing chunk
    ids), written from the community's entities and relations, all communities in one vLLM batch. The output is constrained by a JSON schema (structured outputs).
    """

    SYNTHESIS_PROMPT = (
        "Voici des entités et des relations extraites d'un corpus de documents. Elles forment un même thème.\n"
        "Chaque ligne est précédée, entre crochets, des identifiants des extraits d'où elle provient.\n\n"
        "<entites>\n{entities}\n</entites>\n\n"
        "<relations>\n{relations}\n</relations>\n\n"
        "Rédige en français une synthèse succincte de ce thème, uniquement à partir de ces lignes. "
        "Ne retiens que l'essentiel, c'est-à-dire ce qu'il faut savoir pour comprendre le thème : les acteurs "
        "principaux, ce qu'ils font et les points saillants (décisions, critiques, chiffres clés). Omets les "
        "détails secondaires. Fais le moins de points possible, {max_points} au plus : regroupe les idées "
        "proches en un seul point d'une phrase. N'invente rien.\n\n"
        "Réponds avec quatre champs :\n"
        "- `title` : un titre court pour le thème ;\n"
        "- `opening` : un paragraphe d'ouverture de 2 à 3 phrases donnant le message d'ensemble ;\n"
        "- `points` : les idées essentielles, chacune avec `fact` (l'idée, en une phrase) et `sources` "
        "(les identifiants des extraits des lignes qui l'appuient) ;\n"
        "- `divergences` : les contradictions entre lignes. Une divergence existe seulement si deux "
        "affirmations ne peuvent pas être vraies en même temps : des valeurs, dates ou noms différents pour "
        "la même chose, ou des positions opposées sur une même question. Des valeurs identiques ne sont pas "
        "une divergence. Ne répète jamais une divergence. Ne sont PAS des divergences : qu'une ligne ne "
        "mentionne pas un fait ; qu'un acteur approuve, reprend, confirme ou complète ce que dit un autre ; "
        "une simple différence de formulation. Liste vide s'il n'y en a pas."
    )

    MAX_POINTS = 20  # points per community synthesis

    SOURCED_FACT_SCHEMA = {
        "type": "object",
        "properties": {
            "fact": {"type": "string"},
            "sources": {"type": "array", "minItems": 1, "maxItems": 20, "items": {"type": "integer"}},  # max: anti-loop guard
        },
        "required": ["fact", "sources"],
        "additionalProperties": False,
    }

    SYNTHESIS_SCHEMA = {
        "type": "object",
        "properties": {
            "title": {"type": "string"},
            "opening": {"type": "string"},
            "points": {"type": "array", "minItems": 1, "maxItems": MAX_POINTS, "items": SOURCED_FACT_SCHEMA},
            "divergences": {"type": "array", "maxItems": MAX_POINTS, "items": SOURCED_FACT_SCHEMA},  # max: anti-loop guard
        },
        "required": ["title", "opening", "points", "divergences"],
        "additionalProperties": False,
    }

    def __init__(self, entities: list[EntityNode], relations: list[Relation], llm_model: str | Path,
                 max_cluster_size: int = 400, min_community_size: int = 10, min_relative_size: float = 0.1,
                 max_input_tokens: int = 16384, max_new_tokens: int = 16384,
                 tensor_parallel_size: int = 1, gpu_memory_utilization: float = 0.90,
                 random_state: int = 224) -> None:
        # LLM for community syntheses
        self.llm = LLM(
            model=str(llm_model),
            dtype="bfloat16",
            tensor_parallel_size=tensor_parallel_size,
            gpu_memory_utilization=gpu_memory_utilization,
            max_model_len=max_input_tokens + max_new_tokens,
            max_num_seqs=256,
            trust_remote_code=False,
        )
        self.max_new_tokens = max_new_tokens

        # clustering config
        self.max_cluster_size = max_cluster_size      # Leiden splits any community bigger than this
        self.min_community_size = min_community_size  # communities with fewer entities are dropped
        self.min_relative_size = min_relative_size    # communities smaller than this share of the biggest are dropped
        self.random_state = random_state

        # output storage
        self.graph = nx.Graph()
        self.communities: list[set[str]] = []  # entity keys of each community, biggest first
        self.syntheses: list[dict] = []        # one LLM synthesis per community, same order
        self._build_graph(entities, relations)

    def _generate_json(self, prompts: list[str], schema: dict) -> list[dict]:
        """Run all prompts in one vLLM batch; each output is constrained by the JSON schema."""
        messages = [[{"role": "user", "content": prompt}] for prompt in prompts]
        # for syntheses we want faithful, reproducible text, not variety
        params = SamplingParams(temperature=0.0, max_tokens=self.max_new_tokens,
                                structured_outputs=StructuredOutputsParams(json=schema))
        outputs = self.llm.chat(messages, params)

        input_lengths = [len(o.prompt_token_ids) for o in outputs]
        print(f"{len(outputs)} requests, input tokens: max {max(input_lengths)}, total {sum(input_lengths)}")

        results = []
        for output in outputs:
            generation = output.outputs[0]
            if generation.finish_reason != "stop":  # truncated JSON cannot be parsed
                tail = generation.text[-3000:]
                raise RuntimeError(f"Generation ended with '{generation.finish_reason}' after "
                                   f"{len(generation.token_ids)} tokens "
                                   f"(input {len(output.prompt_token_ids)} tokens). Tail:\n{tail!r}")
            results.append(json.loads(generation.text))
        return results

    @staticmethod
    def key(name: str) -> str:
        """Node id of an entity, insensitive to case."""
        return name.strip().lower()

    def _build_graph(self, entities: list[EntityNode], relations: list[Relation]) -> None:
        """One node per entity, one weighted edge per pair of related entities."""
        # group entities that have the same keys
        mentions: dict[str, list[EntityNode]] = defaultdict(list)
        for e in entities:
            mentions[self.key(e.name)].append(e)

        # combine entities with same keys into a single entity in the knowledge graph
        for entity_key, entity_mentions in mentions.items():
            types = Counter(mention.type for mention in entity_mentions)
            self.graph.add_node(
                entity_key,
                name=entity_mentions[0].name,      # name as first written
                type=types.most_common(1)[0][0],   # most frequent type
                descriptions=list(dict.fromkeys(m.description for m in entity_mentions if m.description)),  # unique, in order
                chunk_ids=sorted({m.chunk_id for m in entity_mentions}),  # all chunks mentioning the entity
            )

        # add relations as edges in the graph
        for r in relations:
            source, target = self.key(r.source), self.key(r.target)
            if source not in self.graph or target not in self.graph or source == target:
                continue  # relation to an entity that was not extracted, or to itself: skipped
            if not self.graph.has_edge(source, target):
                self.graph.add_edge(source, target, weight=0, labels=[], descriptions=[], chunk_ids=[])
            edge = self.graph.edges[source, target]
            edge["weight"] += 1  # how many times this relation was extracted
            edge["labels"].append(r.label)
            if r.description and r.description not in edge["descriptions"]:
                edge["descriptions"].append(r.description)
            if r.chunk_id not in edge["chunk_ids"]:
                edge["chunk_ids"].append(r.chunk_id)

    def build_communities(self) -> None:
        """Hierarchical Leiden on the weighted graph; each entity keeps its finest community.
        Communities with fewer than min_community_size entities, or smaller than min_relative_size of the
        biggest one, are dropped."""
        connected = self.graph.subgraph([n for n in self.graph if self.graph.degree(n) > 0])  # no isolated entity
        clusters = hierarchical_leiden(connected, max_cluster_size=self.max_cluster_size,
                                       random_seed=self.random_state)

        # get only the final communities
        groups: dict[int, set[str]] = defaultdict(set)
        for c in clusters:
            if c.is_final_cluster:
                groups[c.cluster].add(c.node)
        communities = sorted(groups.values(), key=len, reverse=True)
        biggest = len(communities[0]) if communities else 0
        threshold = max(self.min_community_size, self.min_relative_size * biggest)
        self.communities = [g for g in communities if len(g) >= threshold]
        print(f"{len(self.communities)} communities kept, {len(communities) - len(self.communities)} dropped "
              f"(fewer than {threshold:.0f} entities: min {self.min_community_size}, "
              f"or {self.min_relative_size:.0%} of the biggest, {biggest}); "
              f"sizes: {[len(c) for c in self.communities]}")

    @staticmethod
    def _ids(chunk_ids: list[int], max_ids: int = 10) -> str:
        """Chunk ids shown in front of a line (at most max_ids, to keep hub entities short)."""
        return "[" + ", ".join(map(str, sorted(chunk_ids)[:max_ids])) + "]"

    def describe_community(self, community: set[str], max_lines: int = 60) -> tuple[str, str]:
        """Entity and relation lines of a community, each with its chunk ids, most connected first,
        to stay within the prompt budget."""
        nodes = sorted(community, key=self.graph.degree, reverse=True)[:max_lines]
        entity_lines = [f"{self._ids(self.graph.nodes[n]['chunk_ids'])} {self.graph.nodes[n]['name']} "
                        f"({self.graph.nodes[n]['type']}) : {' '.join(self.graph.nodes[n]['descriptions'][:2])}"
                        for n in nodes]

        edges = sorted(self.graph.subgraph(community).edges(data=True),
                       key=lambda edge: edge[2]["weight"], reverse=True)[:max_lines]
        relation_lines = [f"{self._ids(d['chunk_ids'])} {self.graph.nodes[s]['name']} -> {self.graph.nodes[t]['name']} -> "
                          f"{Counter(d['labels']).most_common(1)[0][0]} -> {' '.join(d['descriptions'][:2])}"
                          for s, t, d in edges]
        return "\n".join(entity_lines), "\n".join(relation_lines)

    def synthesize_communities(self) -> None:
        """One LLM synthesis per community, all communities in one vLLM batch."""
        prompts = []
        for community in self.communities:
            entities, relations = self.describe_community(community)
            prompts.append(self.SYNTHESIS_PROMPT.format(entities=entities, relations=relations,
                                                        max_points=self.MAX_POINTS))
        self.syntheses = self._generate_json(prompts, self.SYNTHESIS_SCHEMA) if prompts else []

        for community, synthesis in zip(self.communities, self.syntheses):
            allowed = set(self.community_chunks(community))
            for key in ("points", "divergences"):  # drop duplicate ids and ids outside the community
                for item in synthesis[key]:
                    item["sources"] = [i for i in dict.fromkeys(item["sources"]) if i in allowed]
                synthesis[key] = [item for item in synthesis[key] if item["sources"]]

    def community_chunks(self, community: set[str]) -> list[int]:
        """Chunks where the community's entities are mentioned, to cite its sources with `infos`."""
        return sorted({i for n in community for i in self.graph.nodes[n]["chunk_ids"]})

    def save(self, out_dir: str | Path) -> None:
        """Save the graph and the communities with their syntheses."""
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        graph_data = nx.node_link_data(self.graph, edges="edges")
        (out_dir / "graph.json").write_text(json.dumps(graph_data, ensure_ascii=False, indent=2), encoding="utf-8")
        communities = [{"title": syn["title"], "entities": sorted(c), "chunk_ids": self.community_chunks(c),
                        "opening": syn["opening"], "points": syn["points"], "divergences": syn["divergences"]}
                       for c, syn in zip(self.communities, self.syntheses)]
        (out_dir / "communities.json").write_text(json.dumps(communities, ensure_ascii=False, indent=2),
                                                  encoding="utf-8")

In [ ]:
from pathlib import Path

from graphrag_extractor import GraphRAGExtractor


OUT_DIR = Path("data/INJA/Livrable 1/graphrag")

# 1. Load the saved extractions (no LLM needed: the extractor is not created)
entities, relations = GraphRAGExtractor.load(OUT_DIR / "extractions.jsonl")
print(len(entities), "entity mentions |", len(relations), "relation mentions")

# 2. Build the graph (GraphRAGStore also loads Qwen with vLLM, used later for the syntheses)
store = GraphRAGStore(entities,
                      relations,
                      llm_model="/lustre/fsmisc/dataset/HuggingFace_Models/Qwen/Qwen3.8-27B",
                      tensor_parallel_size=2)
graph = store.graph
print(graph.number_of_nodes(), "entities |", graph.number_of_edges(), "edges")

# 3. Communities and one synthesis per community (all communities in one vLLM batch)
store.build_communities()
store.synthesize_communities()
store.save(OUT_DIR)

for i, (community, synthesis) in enumerate(zip(store.communities, store.syntheses)):
    names = [store.graph.nodes[n]["name"] for n in sorted(community, key=store.graph.degree, reverse=True)[:6]]
    print(f"=== community {i}: {synthesis['title']} ({len(community)} entities: {', '.join(names)})")
    print(synthesis["opening"])
    for point in synthesis["points"]:
        print(f"  • {point['fact']} {point['sources']}")
    for divergence in synthesis["divergences"]:
        print(f"  ⚠ {divergence['fact']} {divergence['sources']}")
    print()